In [2]:

# NovaMart - Day 1
# Task 1: Verify source folders and files

base_path = "Files/project1_novamart-20261107T100452Z-1-001/project1_novamart"

folders = notebookutils.fs.ls(base_path)

for folder in folders:

    print("\nFolder:", folder.name)

    files = notebookutils.fs.ls(folder.path)

    for file in files:
        print(
            "File:", file.name,
            "| Size:", file.size
        )


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 4, Finished, Available, Finished, False)


Folder: ecommerce
File: ecom_orders.jsonl | Size: 4414184

Folder: inventory
File: inventory_snapshot.csv | Size: 4411290

Folder: master
File: customer_changes - Copy.csv | Size: 20613
File: customer_changes.csv | Size: 20613
File: customers.csv | Size: 514556
File: product_price_changes.csv | Size: 7408
File: products.csv | Size: 44028
File: store_targets_FY27.xlsx | Size: 7007
File: stores.csv | Size: 3867

Folder: pos_sales
File: pos_sales_20261001.csv | Size: 75524
File: pos_sales_20261002.csv | Size: 77905
File: pos_sales_20261003.csv | Size: 104289
File: pos_sales_20261004.csv | Size: 109645
File: pos_sales_20261005.csv | Size: 86457
File: pos_sales_20261006.csv | Size: 78613
File: pos_sales_20261007.csv | Size: 68310
File: pos_sales_20261008.csv | Size: 73959
File: pos_sales_20261009.csv | Size: 79543
File: pos_sales_20261010.csv | Size: 112766
File: pos_sales_20261011.csv | Size: 107912
File: pos_sales_20261012.csv | Size: 83087
File: pos_sales_20261013.csv | Size: 76620
File

In [3]:
# Verify POS source files

pos_path = base_path + "/pos_sales"

pos_files = [
    f for f in notebookutils.fs.ls(pos_path)
    if f.name.lower().endswith(".csv")
]

print("Total POS files:", len(pos_files))

for file in sorted(pos_files, key=lambda x: x.name):
    print(file.name)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 5, Finished, Available, Finished, False)

Total POS files: 31
pos_sales_20261001.csv
pos_sales_20261002.csv
pos_sales_20261003.csv
pos_sales_20261004.csv
pos_sales_20261005.csv
pos_sales_20261006.csv
pos_sales_20261007.csv
pos_sales_20261008.csv
pos_sales_20261009.csv
pos_sales_20261010.csv
pos_sales_20261011.csv
pos_sales_20261012.csv
pos_sales_20261013.csv
pos_sales_20261014.csv
pos_sales_20261015.csv
pos_sales_20261016.csv
pos_sales_20261017.csv
pos_sales_20261018.csv
pos_sales_20261019.csv
pos_sales_20261020.csv
pos_sales_20261021.csv
pos_sales_20261022.csv
pos_sales_20261023.csv
pos_sales_20261024.csv
pos_sales_20261025.csv
pos_sales_20261026.csv
pos_sales_20261027.csv
pos_sales_20261028.csv
pos_sales_20261029.csv
pos_sales_20261030.csv
pos_sales_20261031.csv


In [4]:
# Step 1: Define POS folder path
pos_path = base_path + "/pos_sales"

# Step 2: Get all CSV files

pos_files = [
    f.path
    for f in notebookutils.fs.ls(pos_path)
    if f.name.lower().endswith(".csv")
]
# Step 3: Verify file count

print("Total POS files:", len(pos_files))


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 6, Finished, Available, Finished, False)

Total POS files: 31


In [5]:

from pyspark.sql import functions as F

# Read all POS CSV files together

pos_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .csv(pos_files)
)

# Add source filename for auditing

pos_df = pos_df.withColumn(
    "source_file",
    F.regexp_extract(
        F.input_file_name(),
        r"([^/]+)$",
        1
    )
)
print("POS Data Loaded Successfully")

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 7, Finished, Available, Finished, False)

POS Data Loaded Successfully


In [6]:
pos_df.printSchema()

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 8, Finished, Available, Finished, False)

root
 |-- txn_id: string (nullable = true)
 |-- line_no: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- txn_ts: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_pct: string (nullable = true)
 |-- pos_terminal: string (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- source_file: string (nullable = false)



In [7]:
pos_df.show(10, truncate=False)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 9, Finished, Available, Finished, False)

+-----------------------+-------+--------+-------------------+-----------+----------+--------+----------+------------+------------+------------+-----------------------------------------------------------+
|txn_id                 |line_no|store_id|txn_ts             |customer_id|product_id|quantity|unit_price|discount_pct|pos_terminal|payment_mode|source_file                                                |
+-----------------------+-------+--------+-------------------+-----------+----------+--------+----------+------------+------------+------------+-----------------------------------------------------------+
|POS-S002-20261024-00009|1      |S002    |2026-10-24 11:59:26|NULL       |P0445     |5       |360       |5           |T02         |Card        |pos_sales_20261024.csv?version=1794295086000?flength=168647|
|POS-S038-20261024-00027|6      |S038    |2026-10-24 16:58:45|NULL       |P0105     |1       |340       |0           |T08         |UPI         |pos_sales_20261024.csv?version=17942

In [8]:
total_rows = pos_df.count()
total_columns = len(pos_df.columns)
print("Total Rows:", total_rows)
print("Total Columns:", total_columns)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 10, Finished, Available, Finished, False)

Total Rows: 39650
Total Columns: 12


In [9]:

from pyspark.sql import functions as F

total_records = pos_df.count()

null_results = []

for column in pos_df.columns:

    null_count = pos_df.filter(
        F.col(column).isNull() |
        (F.trim(F.col(column)) == "")
    ).count()

    null_percentage = (
        (null_count / total_records) * 100
        if total_records > 0 else 0
    )

    null_results.append((
        column,
        null_count,
        round(null_percentage, 2)
    ))

null_df = spark.createDataFrame(
    null_results,
    ["column_name", "null_count", "null_percentage"]
)

display(null_df.orderBy(F.desc("null_count")))


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 14c899b0-6955-4b3e-8632-cc17e9cdd4ae)

In [10]:

total_records = pos_df.count()

distinct_records = pos_df.dropDuplicates().count()

duplicate_records = total_records - distinct_records

print("Total Records:", total_records)
print("Distinct Records:", distinct_records)
print("Exact Duplicate Records:", duplicate_records)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 12, Finished, Available, Finished, False)

Total Records: 39650
Distinct Records: 39258
Exact Duplicate Records: 392


In [11]:

from pyspark.sql import functions as F

file_null_analysis = (
    pos_df
    .groupBy("source_file")
    .agg(
        F.count("*").alias("total_records"),

        F.sum(
            F.when(F.col("payment_mode").isNull(), 1)
            .otherwise(0)
        ).alias("missing_payment_mode"),

        F.sum(
            F.when(F.col("customer_id").isNull(), 1)
            .otherwise(0)
        ).alias("missing_customer_id")
    )
)

display(file_null_analysis.orderBy("source_file"))


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 420e64c6-ebbc-4e81-b648-0d3299be2008)

In [12]:

# Count unique source files

file_count = pos_df.select("source_file").distinct().count()

print("Unique POS files loaded:", file_count)

# Display complete file names and record counts

file_summary = (
    pos_df
    .groupBy("source_file")
    .agg(
        F.count("*").alias("total_records"),
        F.sum(
            F.when(
                F.col("payment_mode").isNull() |
                (F.trim(F.col("payment_mode")) == ""),
                1
            ).otherwise(0)
        ).alias("missing_payment_mode")
    )
    .orderBy("source_file")
)

display(file_summary)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 14, Finished, Available, Finished, False)

Unique POS files loaded: 31


SynapseWidget(Synapse.DataFrame, 9ff2a6d2-3708-49bb-ad29-9f992f3ab166)

In [13]:

# 1. Total records in POS DataFrame
total_records = pos_df.count()

# 2. Count unique source files
unique_files = pos_df.select(
    "source_file"
).distinct().count()

# 3. Count records with missing source_file
missing_source = pos_df.filter(
    F.col("source_file").isNull()
).count()

# 4. Sum records grouped by source file
grouped_total = (
    pos_df
    .groupBy("source_file")
    .count()
    .agg(F.sum("count").alias("total"))
    .first()["total"]
)

print("Total DataFrame Records:", total_records)
print("Unique Source Files:", unique_files)
print("Records with Missing Source:", missing_source)
print("Grouped Records Total:", grouped_total)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 15, Finished, Available, Finished, False)

Total DataFrame Records: 39650
Unique Source Files: 31
Records with Missing Source: 0
Grouped Records Total: 39650


In [14]:

print("Files discovered:", len(pos_files))

for file in pos_files:
    print(file.split("/")[-1])


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 16, Finished, Available, Finished, False)

Files discovered: 31
pos_sales_20261001.csv
pos_sales_20261002.csv
pos_sales_20261003.csv
pos_sales_20261004.csv
pos_sales_20261005.csv
pos_sales_20261006.csv
pos_sales_20261007.csv
pos_sales_20261008.csv
pos_sales_20261009.csv
pos_sales_20261010.csv
pos_sales_20261011.csv
pos_sales_20261012.csv
pos_sales_20261013.csv
pos_sales_20261014.csv
pos_sales_20261015.csv
pos_sales_20261016.csv
pos_sales_20261017.csv
pos_sales_20261018.csv
pos_sales_20261019.csv
pos_sales_20261020.csv
pos_sales_20261021.csv
pos_sales_20261022.csv
pos_sales_20261023.csv
pos_sales_20261024.csv
pos_sales_20261025.csv
pos_sales_20261026.csv
pos_sales_20261027.csv
pos_sales_20261028.csv
pos_sales_20261029.csv
pos_sales_20261030.csv
pos_sales_20261031.csv


In [15]:
header_results = []

for path in pos_files:

    # Read the first line of each CSV file
    header = spark.read.text(path).first()[0]

    # Extract filename
    file_name = path.split("/")[-1].split("?")[0]

    header_results.append((file_name, header))

header_df = spark.createDataFrame(
    header_results,
    ["file_name", "csv_header"]
)

display(header_df.orderBy("file_name"))


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e5b8e34a-7c0f-4bc1-9ab5-4f3c16812253)

In [16]:

from pyspark.sql import functions as F

# Analyze payment mode values

payment_analysis = (
    pos_df
    .groupBy("payment_mode")
    .agg(
        F.count("*").alias("total_records")
    )
    .orderBy(F.desc("total_records"))
)

display(payment_analysis)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8e814eca-d2e5-4155-a2d6-8e68e09cca21)

In [17]:

payment_by_file = (
    pos_df
    .groupBy("source_file")
    .agg(
        F.count("*").alias("total_records"),
        F.count("payment_mode").alias("populated_payment_mode"),
        F.sum(
            F.when(
                F.col("payment_mode").isNull(), 1
            ).otherwise(0)
        ).alias("null_payment_mode")
    )
    .orderBy("source_file")
)

display(payment_by_file)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 794307c5-b1ab-412a-8ac8-41819e1f7ed0)

In [18]:

from pyspark.sql import functions as F

# Columns to investigate
check_columns = [
    "txn_id",
    "store_id",
    "txn_ts",
    "customer_id",
    "product_id",
    "quantity",
    "unit_price",
    "discount_pct",
    "pos_terminal",
    "payment_mode"
]

# Calculate NULL count and percentage
total_records = pos_df.count()

quality_results = []

for column in check_columns:

    null_count = pos_df.filter(
        F.col(column).isNull() |
        (F.trim(F.col(column)) == "")
    ).count()

    null_percentage = (null_count / total_records) * 100

    quality_results.append((
        column,
        null_count,
        round(null_percentage, 2)
    ))

quality_df = spark.createDataFrame(
    quality_results,
    ["column_name", "missing_count", "missing_percentage"]
)

display(quality_df.orderBy(F.desc("missing_count")))


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 54dbcafa-3557-43af-82e4-b171be94ea38)

In [19]:

from pyspark.sql import functions as F

# Analyze missing customer IDs by source file
customer_analysis = (
    pos_df
    .groupBy("source_file")
    .agg(
        F.count("*").alias("total_records"),
        F.sum(
            F.when(F.col("customer_id").isNull(), 1)
             .otherwise(0)
        ).alias("missing_customer_id")
    )
    .withColumn(
        "missing_percentage",
        F.round(
            F.col("missing_customer_id") /
            F.col("total_records") * 100, 2
        )
    )
    .orderBy(F.desc("missing_customer_id"))
)

display(customer_analysis)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1c75af96-c549-4045-8852-7ce94057c8d5)

In [20]:

from pyspark.sql import functions as F

distinct_analysis = (
    pos_df
    .groupBy("source_file")
    .agg(
        F.count("*").alias("total_records"),
        F.countDistinct("txn_id").alias("distinct_transactions"),
        F.countDistinct("store_id").alias("distinct_stores"),
        F.countDistinct("customer_id").alias("distinct_customers"),
        F.countDistinct("product_id").alias("distinct_products")
    )
    .orderBy("source_file")
)

display(distinct_analysis)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 169cd45d-9c19-4b1d-af54-dd403a5a62f7)

In [21]:
duplicate_keys = (
    pos_df
    .groupBy("source_file", "txn_id", "line_no")
    .agg(
        F.count("*").alias("record_count")
    )
    .filter(
        F.col("record_count") > 1
    )
)

duplicate_summary = (
    duplicate_keys
    .groupBy("source_file")
    .agg(
        F.count("*").alias("duplicate_key_groups"),
        F.sum(
            F.col("record_count") - 1
        ).alias("extra_duplicate_rows")
    )
    .orderBy("source_file")
)

display(duplicate_summary)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f9015ca5-ac1f-4625-8840-5f3ea8e2625c)

In [22]:
duplicate_records = (
    pos_df
    .join(
        duplicate_keys.select(
            "source_file",
            "txn_id",
            "line_no"
        ),
        on=["source_file", "txn_id", "line_no"],
        how="inner"
    )
    .orderBy("source_file", "txn_id", "line_no")
)

display(duplicate_records.limit(20))

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 210d888c-a871-4565-a8ab-31bd56b386e1)

In [23]:

from pyspark.sql import functions as F

date_analysis = (
    pos_df
    .withColumn(
        "transaction_date",
        F.to_timestamp("txn_ts")
    )
    .groupBy("source_file")
    .agg(
        F.min("transaction_date").alias("min_date"),
        F.max("transaction_date").alias("max_date"),
        F.sum(
            F.when(F.col("transaction_date").isNull(), 1)
             .otherwise(0)
        ).alias("invalid_timestamps"),
        F.count("*").alias("total_records")
    )
    .orderBy("source_file")
)

display(date_analysis)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 24c82ca6-33b3-449e-a2ca-5a296aa3c3f3)

In [24]:

# List all files inside the Inventory folder

def find_inventory_files(path):
    for item in notebookutils.fs.ls(path):
        if item.isDir:
            find_inventory_files(item.path)
        elif "inventory" in item.path.lower():
            print(item.path)

find_inventory_files("Files/")


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 26, Finished, Available, Finished, False)

abfss://a3dec072-153b-40cc-ae81-06c66b7d6cfc@onelake.dfs.fabric.microsoft.com/01c82466-9006-4134-b795-5fce4a6f12ea/Files/project1_novamart-20261107T100452Z-1-001/project1_novamart/inventory/inventory_snapshot.csv


In [25]:

# Read Inventory CSV file

inventory_path = None

def find_inventory_file(path):
    global inventory_path

    for item in notebookutils.fs.ls(path):
        if item.isDir:
            find_inventory_file(item.path)
        elif "inventory_snapshot.csv" in item.path:
            inventory_path = item.path

find_inventory_file("Files/")

# Load CSV into DataFrame
inventory_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(inventory_path)
)

# Display first 10 records
display(inventory_df.limit(10))


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 147d9580-e685-4f22-9f93-b016ee0ba295)

In [26]:

# Calculate total rows and columns

total_rows = inventory_df.count()
total_columns = len(inventory_df.columns)

print("Total Rows:", total_rows)
print("Total Columns:", total_columns)

print("Column Names:")
print(inventory_df.columns)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 28, Finished, Available, Finished, False)

Total Rows: 155000
Total Columns: 5
Column Names:
['snapshot_date', 'store_id', 'product_id', 'on_hand_qty', 'reorder_level']


In [27]:
from pyspark.sql import functions as F

# Check missing values in each column

total_records = inventory_df.count()

null_results = []

for column in inventory_df.columns:

    null_count = inventory_df.filter(
        F.col(column).isNull()
    ).count()

    null_percentage = (
        null_count / total_records
    ) * 100

    null_results.append((
        column,
        null_count,
        round(null_percentage, 2)
    ))

null_df = spark.createDataFrame(
    null_results,
    ["column_name", "null_count", "null_percentage"]
)

display(null_df.orderBy(F.desc("null_count")))


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d596af88-7ccf-410e-9c71-9a53216dce91)

In [28]:
# Check duplicate records

total_records = inventory_df.count()

distinct_records = inventory_df.dropDuplicates().count()

duplicate_records = total_records - distinct_records

print("Total Records:", total_records)
print("Distinct Records:", distinct_records)
print("Duplicate Records:", duplicate_records)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 30, Finished, Available, Finished, False)

Total Records: 155000
Distinct Records: 155000
Duplicate Records: 0


In [29]:

from pyspark.sql import functions as F

# Inventory business validation

inventory_quality = inventory_df.agg(

    F.sum(
        F.when(F.col("on_hand_qty") < 0, 1).otherwise(0)
    ).alias("negative_stock"),

    F.sum(
        F.when(F.col("reorder_level") < 0, 1).otherwise(0)
    ).alias("negative_reorder_level"),

    F.sum(
        F.when(F.col("on_hand_qty") < F.col("reorder_level"), 1).otherwise(0)
    ).alias("below_reorder_level"),

    F.sum(
        F.when(F.col("on_hand_qty") == 0, 1).otherwise(0)
    ).alias("zero_stock")

)

display(inventory_quality)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 84c35e09-7640-40ff-9f8c-172933994de8)

In [30]:
# Identify negative stock records

negative_stock_df = inventory_df.filter(
    F.col("on_hand_qty") < 0
)

display(negative_stock_df)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 26203a89-01fa-4289-831b-0a930f9a0989)

In [31]:
# Identify products below reorder level

below_reorder_df = inventory_df.filter(
    F.col("on_hand_qty") < F.col("reorder_level")
)

display(below_reorder_df)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 57b2cd37-b94d-4a4d-aa8b-2a20e120120b)

In [32]:
# Identify zero-stock products

zero_stock_df = inventory_df.filter(
    F.col("on_hand_qty") == 0
)

display(zero_stock_df)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9a6b0bdc-ae20-4f1d-a6a9-4ebfd0691263)

In [33]:
# Calculate stock availability

total_inventory = inventory_df.count()

available_stock = inventory_df.filter(
    F.col("on_hand_qty") > 0
).count()

zero_stock = inventory_df.filter(
    F.col("on_hand_qty") == 0
).count()

availability_percentage = (
    available_stock / total_inventory
) * 100

print("Total Inventory Records:", total_inventory)
print("Available Stock Records:", available_stock)
print("Zero Stock Records:", zero_stock)
print("Stock Availability Percentage:", round(availability_percentage, 2))

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 35, Finished, Available, Finished, False)

Total Inventory Records: 155000
Available Stock Records: 151817
Zero Stock Records: 3028
Stock Availability Percentage: 97.95


In [34]:
# Final Inventory Quality Summary

inventory_summary = inventory_df.agg(
    F.count("*").alias("total_records"),

    F.sum(
        F.when(F.col("on_hand_qty") < 0, 1).otherwise(0)
    ).alias("negative_stock"),

    F.sum(
        F.when(F.col("on_hand_qty") == 0, 1).otherwise(0)
    ).alias("zero_stock"),

    F.sum(
        F.when(
            F.col("on_hand_qty") < F.col("reorder_level"), 1
        ).otherwise(0)
    ).alias("below_reorder_level")
)

display(inventory_summary)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 27663a3d-d362-47f2-ab9d-6548431caf81)

In [35]:

# List all files inside the E-commerce folder

def find_ecommerce_files(path):
    for item in notebookutils.fs.ls(path):
        if item.isDir:
            find_ecommerce_files(item.path)
        elif "ecommerce" in item.path.lower():
            print(item.path)

find_ecommerce_files("Files/")


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 37, Finished, Available, Finished, False)

abfss://a3dec072-153b-40cc-ae81-06c66b7d6cfc@onelake.dfs.fabric.microsoft.com/01c82466-9006-4134-b795-5fce4a6f12ea/Files/project1_novamart-20261107T100452Z-1-001/project1_novamart/ecommerce/ecom_orders.jsonl


In [36]:

# Step 1: Find the E-commerce JSON file

ecommerce_files = []

def collect_ecommerce_files(path):
    for item in notebookutils.fs.ls(path):
        if item.isDir:
            collect_ecommerce_files(item.path)
        elif "ecom_orders" in item.path.lower():
            ecommerce_files.append(item.path)

collect_ecommerce_files("Files/")

# Step 2: Read JSON data

ecommerce_df = spark.read.json(ecommerce_files[0])

# Step 3: Display records

display(ecommerce_df.limit(10))

# Step 4: Check structure

ecommerce_df.printSchema()

# Step 5: Count records

print("Total E-commerce Records:", ecommerce_df.count())


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d7bd7b71-b5cc-494c-8416-2df5af25b4a2)

root
 |-- channel: string (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- delivery_fee: long (nullable = true)
 |-- device: struct (nullable = true)
 |    |-- app_version: string (nullable = true)
 |    |-- os: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- discount_pct: long (nullable = true)
 |    |    |-- product_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)
 |    |    |-- unit_price: long (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- payment: struct (nullable = true)
 |    |-- gateway_ref: string (nullable = true)
 |    |-- mode: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- status: string (nullable = true)

Total E-commerce Records: 9380


In [37]:

# E-commerce missing value analysis

from pyspark.sql import functions as F

total_records = ecommerce_df.count()

null_analysis = ecommerce_df.select(
    *[
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in ecommerce_df.columns
    ]
)

display(null_analysis)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ddf98815-270d-4c24-bcd2-8575eb97410e)

In [38]:

# Extract individual items from the items array

from pyspark.sql import functions as F

ecommerce_items_df = ecommerce_df.withColumn(
    "item",
    F.explode_outer("items")
)

# Select important item-level columns

ecommerce_items_df = ecommerce_items_df.select(
    "order_id",
    "customer_id",
    "ship_city",
    "status",
    F.col("item.product_id").alias("product_id"),
    F.col("item.qty").alias("quantity"),
    F.col("item.unit_price").alias("unit_price"),
    F.col("item.discount_pct").alias("discount_pct")
)

display(ecommerce_items_df)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 988f86a5-680c-49ea-bb95-f6950887131c)

In [39]:
# Check duplicate order IDs

duplicate_orders = (
    ecommerce_df
    .groupBy("order_id")
    .agg(
        F.count("*").alias("record_count")
    )
    .filter(F.col("record_count") > 1)
    .orderBy(F.desc("record_count"))
)

display(duplicate_orders)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 41, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 094084cf-2385-4942-9511-1bd1b2b61844)

In [40]:
# Display duplicate order details

duplicate_order_details = (
    ecommerce_df
    .join(
        duplicate_orders.select("order_id"),
        on="order_id",
        how="inner"
    )
    .orderBy("order_id")
)

display(duplicate_order_details)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0cd5fa5f-c3bb-4b2b-9035-ce5a9e6ab875)

In [41]:
# Count exact duplicate records

total_records = ecommerce_df.count()

distinct_records = ecommerce_df.dropDuplicates().count()

exact_duplicates = total_records - distinct_records

print("Total Records:", total_records)
print("Distinct Records:", distinct_records)
print("Exact Duplicate Records:", exact_duplicates)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 43, Finished, Available, Finished, False)

Total Records: 9380
Distinct Records: 9334
Exact Duplicate Records: 46


In [42]:
# Inspect order timestamps

ecommerce_df.select(
    "order_id",
    "order_ts"
).show(20, truncate=False)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 44, Finished, Available, Finished, False)

+---------------+-------------------------+
|order_id       |order_ts                 |
+---------------+-------------------------+
|EC2026102500312|2026-10-25T04:32:19+05:30|
|EC2026102100234|2026-10-21T14:21:58+05:30|
|EC2026100800106|08/10/2026 16:29         |
|EC2026101700270|2026-10-17T02:58:29+05:30|
|EC2026100400077|2026-10-04T08:13:03+05:30|
|EC2026102800200|2026-10-28T15:52:43+05:30|
|EC2026103000017|2026-10-30T18:11:04+05:30|
|EC2026100700060|2026-10-07T15:55:34+05:30|
|EC2026100400178|2026-10-04T04:08:19+05:30|
|EC2026101000032|2026-10-10T04:47:52+05:30|
|EC2026102300186|2026-10-23T10:03:32+05:30|
|EC2026101800294|2026-10-18T13:41:28+05:30|
|EC2026101800012|2026-10-18T23:01:13+05:30|
|EC2026100100142|2026-10-01T10:40:55+05:30|
|EC2026102900218|2026-10-29T21:44:03+05:30|
|EC2026100500183|2026-10-05T07:28:46+05:30|
|EC2026102600112|26/10/2026 07:11         |
|EC2026100600082|2026-10-06T02:04:19+05:30|
|EC2026102200355|2026-10-22T02:08:16+05:30|
|EC2026101000142|2026-10-10T11:5

In [43]:
# Check column data types

ecommerce_df.select("order_ts").printSchema()

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 45, Finished, Available, Finished, False)

root
 |-- order_ts: string (nullable = true)



In [44]:
# Standardize different timestamp formats

ecommerce_df = ecommerce_df.withColumn(
    "parsed_order_ts",
    F.coalesce(
        F.to_timestamp("order_ts", "yyyy-MM-dd'T'HH:mm:ssXXX"),
        F.to_timestamp("order_ts", "yyyy-MM-dd'T'HH:mmXXX"),
        F.to_timestamp("order_ts", "dd/MM/yyyy HH:mm"),
        F.to_timestamp("order_ts", "yyyy-MM-dd'T'HH:mm:ss"),
        F.to_timestamp("order_ts", "yyyy-MM-dd'T'HH:mm")
    )
)

display(
    ecommerce_df.select(
        "order_id",
        "order_ts",
        "parsed_order_ts"
    )
)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 46, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 034ceba1-1e4c-4c82-98cd-335fe944e2eb)

In [45]:
# Check timestamp quality

timestamp_quality = ecommerce_df.agg(
    F.count("*").alias("total_records"),

    F.sum(
        F.when(F.col("order_ts").isNull(), 1).otherwise(0)
    ).alias("original_null_timestamps"),

    F.sum(
        F.when(
            F.col("order_ts").isNotNull() &
            F.col("parsed_order_ts").isNull(), 1
        ).otherwise(0)
    ).alias("invalid_timestamps")
)

display(timestamp_quality)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3caa0040-d62f-466c-be34-0bef16da72e0)

In [46]:
# Analyze order status

status_analysis = ecommerce_df.groupBy("status") \
    .count() \
    .orderBy(F.desc("count"))

display(status_analysis)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 48, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3d33b789-d71d-4446-b5b5-0da671938912)

In [47]:
# Analyze sales channels

channel_analysis = ecommerce_df.groupBy("channel") \
    .count() \
    .orderBy(F.desc("count"))

display(channel_analysis)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 49, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 348225a8-7dee-4cd8-a70c-1e78186b99f8)

In [48]:

# Analyze payment modes

payment_analysis = (
    ecommerce_df
    .groupBy(F.col("payment.mode").alias("payment_mode"))
    .count()
    .orderBy(F.desc("count"))
)

display(payment_analysis)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 50, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f2ed82ee-fcd2-47aa-ab40-54c9fc1fd8bb)

In [49]:

# Analyze orders by shipping city

city_analysis = (
    ecommerce_df
    .groupBy("ship_city")
    .count()
    .orderBy(F.desc("count"))
)

display(city_analysis)


StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 51, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ac05d155-935b-4480-9cbf-e439c18cea66)

In [50]:
# Calculate revenue for each product item

ecommerce_items_df = ecommerce_items_df.withColumn(
    "gross_amount",
    F.col("quantity") * F.col("unit_price")
)

ecommerce_items_df = ecommerce_items_df.withColumn(
    "net_amount",
    F.col("gross_amount") * 
    (1 - F.col("discount_pct") / 100)
)

display(ecommerce_items_df)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 52, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f4575485-cff0-483e-bf74-733225ff1863)

In [51]:
# Calculate overall revenue summary

revenue_summary = ecommerce_items_df.agg(
    F.sum("gross_amount").alias("total_gross_revenue"),
    F.sum("net_amount").alias("total_net_revenue"),
    F.sum(
        F.col("gross_amount") - F.col("net_amount")
    ).alias("total_discount")
)

display(revenue_summary)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 53, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4aaaaa15-86eb-404d-883d-f9becdffad08)

In [52]:
# Calculate revenue by shipping city

city_revenue = (
    ecommerce_items_df
    .groupBy("ship_city")
    .agg(
        F.sum("net_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_quantity")
    )
    .orderBy(F.desc("total_revenue"))
)

display(city_revenue)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 54, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7d258da3-6be3-4552-b62c-f57611b00fb7)

In [53]:
# Analyze revenue by order status

status_revenue = (
    ecommerce_items_df
    .groupBy("status")
    .agg(
        F.sum("net_amount").alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders")
    )
    .orderBy(F.desc("total_revenue"))
)

display(status_revenue)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 55, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 385c8191-fac4-4c3b-bc8f-839556f51bb1)

In [54]:
# Select order ID and channel from original orders

channel_df = (
    ecommerce_df
    .select("order_id", "channel")
    .dropDuplicates(["order_id"])
)

# Join channel information with product details

ecommerce_items_channel_df = (
    ecommerce_items_df
    .join(channel_df, on="order_id", how="left")
)

display(ecommerce_items_channel_df)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 56, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8da3ec7d-a0d7-4343-b424-77d43eea6471)

In [55]:
# Analyze revenue by sales channel

channel_revenue = (
    ecommerce_items_channel_df
    .groupBy("channel")
    .agg(
        F.sum("net_amount").alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders")
    )
    .orderBy(F.desc("total_revenue"))
)

display(channel_revenue)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 57, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eb034df2-7b39-433e-abe3-71c81cd6218b)

In [56]:
# Extract payment mode from original ecommerce data

payment_df = ecommerce_df.select(
    "order_id",
    F.col("payment.mode").alias("payment_mode")
).dropDuplicates(["order_id"])

display(payment_df)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 58, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 18dce963-f195-43e0-9cf3-13d442d5fac6)

In [57]:
# Join payment information with sales data

ecommerce_payment_df = ecommerce_items_channel_df.join(
    payment_df,
    on="order_id",
    how="left"
)

display(ecommerce_payment_df)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 59, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c78a4116-086a-4311-9988-68f041151565)

In [58]:
# Analyze revenue by payment mode

payment_revenue = (
    ecommerce_payment_df
    .groupBy("payment_mode")
    .agg(
        F.sum("net_amount").alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders")
    )
    .orderBy(F.desc("total_revenue"))
)

display(payment_revenue)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 60, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 742fd51d-76e3-46cf-8142-0090e8904092)

In [59]:
# Analyze revenue by product

product_revenue = (
    ecommerce_items_df
    .groupBy("product_id")
    .agg(
        F.sum("net_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_quantity")
    )
    .orderBy(F.desc("total_revenue"))
)

display(product_revenue.limit(10))

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 61, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ce7afc37-2fa6-454c-89df-a48781356d3c)

In [60]:
# List master data files

def find_master_files(path):
    for item in notebookutils.fs.ls(path):
        if item.isDir:
            find_master_files(item.path)
        elif "master" in item.path.lower():
            print(item.path)

find_master_files("Files/")

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 62, Finished, Available, Finished, False)

abfss://a3dec072-153b-40cc-ae81-06c66b7d6cfc@onelake.dfs.fabric.microsoft.com/01c82466-9006-4134-b795-5fce4a6f12ea/Files/project1_novamart-20261107T100452Z-1-001/project1_novamart/master/customer_changes - Copy.csv
abfss://a3dec072-153b-40cc-ae81-06c66b7d6cfc@onelake.dfs.fabric.microsoft.com/01c82466-9006-4134-b795-5fce4a6f12ea/Files/project1_novamart-20261107T100452Z-1-001/project1_novamart/master/customer_changes.csv
abfss://a3dec072-153b-40cc-ae81-06c66b7d6cfc@onelake.dfs.fabric.microsoft.com/01c82466-9006-4134-b795-5fce4a6f12ea/Files/project1_novamart-20261107T100452Z-1-001/project1_novamart/master/customers.csv
abfss://a3dec072-153b-40cc-ae81-06c66b7d6cfc@onelake.dfs.fabric.microsoft.com/01c82466-9006-4134-b795-5fce4a6f12ea/Files/project1_novamart-20261107T100452Z-1-001/project1_novamart/master/product_price_changes.csv
abfss://a3dec072-153b-40cc-ae81-06c66b7d6cfc@onelake.dfs.fabric.microsoft.com/01c82466-9006-4134-b795-5fce4a6f12ea/Files/project1_novamart-20261107T100452Z-1-001/p

In [61]:
# Load customer master data
customer_path = "Files/project1_novamart-20261107T100452Z-1-001/project1_novamart/master/customers.csv"
customers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(customer_path)
)

display(customers_df)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 63, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e5356dfb-fe73-4784-8da7-e3d480a7205e)

In [62]:
# Inspect customer data

customers_df.printSchema()

print("Total Customer Records:", customers_df.count())

print("Total Columns:", len(customers_df.columns))

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 64, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)

Total Customer Records: 5000
Total Columns: 11


In [63]:
customers_df.printSchema()

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 65, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)



In [64]:
# Check duplicate customer IDs

duplicate_customers = (
    customers_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

display(duplicate_customers)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 66, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bf996642-c3ff-4cf8-afc9-5d96960a37f8)

In [65]:
# Calculate duplicate customer ID count

total_customers = customers_df.count()

distinct_customers = customers_df.select(
    "customer_id"
).distinct().count()

duplicate_count = total_customers - distinct_customers

print("Total Customers:", total_customers)
print("Distinct Customers:", distinct_customers)
print("Duplicate Customer IDs:", duplicate_count)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 67, Finished, Available, Finished, False)

Total Customers: 5000
Distinct Customers: 5000
Duplicate Customer IDs: 0


In [66]:
# Check NULL values in customer data

from pyspark.sql import functions as F

null_analysis = customers_df.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in customers_df.columns
])

display(null_analysis)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 68, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 44070145-f6cc-4442-b774-43c35016d95d)

In [67]:
# Calculate NULL percentage for each column

total_records = customers_df.count()

null_percentage = customers_df.select([
    (
        F.count(F.when(F.col(c).isNull(), 1)) / total_records * 100
    ).alias(c)
    for c in customers_df.columns
])

display(null_percentage)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 69, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5e5bb6c4-b737-442a-a575-1e97e796ebb6)

In [68]:
# Identify customers with missing email addresses

missing_email_customers = customers_df.filter(
    F.col("email").isNull()
)

display(missing_email_customers)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 70, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9e2d3d5d-fd54-4122-b485-946df7b4160a)

In [69]:
print(
    "Customers with missing emails:",
    missing_email_customers.count()
)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 71, Finished, Available, Finished, False)

Customers with missing emails: 147


In [70]:
# Check duplicate email addresses

duplicate_emails = (
    customers_df
    .filter(F.col("email").isNotNull())
    .groupBy("email")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

display(duplicate_emails)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 72, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9dc76594-cc6d-44ef-814d-bc1990449403)

In [71]:
print(
    "Total Duplicate Email Groups:",
    duplicate_emails.count()
)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 73, Finished, Available, Finished, False)

Total Duplicate Email Groups: 0


In [72]:
# Check invalid email formats

invalid_emails = customers_df.filter(
    F.col("email").isNotNull() &
    (~F.col("email").rlike(
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
    ))
)

print("Invalid Email Records:", invalid_emails.count())

display(invalid_emails)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 74, Finished, Available, Finished, False)

Invalid Email Records: 0


SynapseWidget(Synapse.DataFrame, 253462b6-079f-4185-845d-9fd96b1dc1a4)

In [73]:
# Check invalid phone numbers

invalid_phones = customers_df.filter(
    F.col("phone").isNotNull() &
    (~F.col("phone").cast("string").rlike(r"^[0-9]{10}$"))
)

print("Invalid Phone Records:", invalid_phones.count())

display(invalid_phones)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 75, Finished, Available, Finished, False)

Invalid Phone Records: 0


SynapseWidget(Synapse.DataFrame, 67b2450e-435c-42a9-bd03-f03d26c129e5)

In [74]:
# Check invalid phone numbers

invalid_phones = customers_df.filter(
    F.col("phone").isNotNull() &
    (~F.col("phone").cast("string").rlike(r"^[0-9]{10}$"))
)

print("Invalid Phone Records:", invalid_phones.count())

display(invalid_phones)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 76, Finished, Available, Finished, False)

Invalid Phone Records: 0


SynapseWidget(Synapse.DataFrame, 65a533d8-e3aa-41a2-abcc-f96a187fb9e2)

In [75]:
# Check NULL phone numbers

missing_phones = customers_df.filter(
    F.col("phone").isNull()
)

print("Missing Phone Numbers:", missing_phones.count())

display(missing_phones)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 77, Finished, Available, Finished, False)

Missing Phone Numbers: 0


SynapseWidget(Synapse.DataFrame, ba31d1d5-a047-4534-be70-b9b9d3f2b8b8)

In [76]:
# Analyze gender values

gender_analysis = (
    customers_df
    .groupBy("gender")
    .count()
    .orderBy(F.desc("count"))
)

display(gender_analysis)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 78, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3d176382-22e1-4a15-b292-9082c6b63da5)

In [77]:
# Analyze loyalty tiers

loyalty_analysis = (
    customers_df
    .groupBy("loyalty_tier")
    .count()
    .orderBy(F.desc("count"))
)

display(loyalty_analysis)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 79, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e37e00ee-9f04-429a-a06c-ae761237c3a4)

In [78]:
from pyspark.sql import functions as F

# Standardize customer names

customers_df = (
    customers_df
    .withColumn(
        "first_name",
        F.initcap(F.trim(F.col("first_name")))
    )
    .withColumn(
        "last_name",
        F.initcap(F.trim(F.col("last_name")))
    )
)

display(customers_df.select(
    "customer_id",
    "first_name",
    "last_name"
))

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 80, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 28b76c91-e1c1-4721-b756-b0eeaf788496)

In [79]:
# Standardize email addresses

customers_df = customers_df.withColumn(
    "email",
    F.lower(F.trim(F.col("email")))
)

display(customers_df.select(
    "customer_id",
    "email"
))

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 81, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ee93646f-a9b4-4a98-a96a-e821a2701a0a)

In [80]:
# Standardize city names

customers_df = customers_df.withColumn(
    "city",
    F.initcap(F.trim(F.col("city")))
)

display(
    customers_df.groupBy("city")
    .count()
    .orderBy(F.desc("count"))
)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 82, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 470907af-eaa1-4b37-a9b8-bc9a2a2250ee)

In [81]:
# Standardize gender values

customers_df = customers_df.withColumn(
    "gender",
    F.upper(F.trim(F.col("gender")))
)

display(
    customers_df.groupBy("gender")
    .count()
    .orderBy(F.desc("count"))
)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 83, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7e3e1bdf-a11f-4486-9199-c1c9e490b476)

In [82]:
from pyspark.sql import functions as F

customers_df = customers_df.withColumn(
    "age",
    F.floor(
        F.months_between(
            F.current_date(),
            F.col("birth_date")
        ) / 12
    )
)

display(customers_df.select(
    "customer_id",
    "birth_date",
    "age"
))

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 84, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3623347f-c239-46a3-9f02-a22e81734dae)

In [83]:
age_analysis = customers_df.withColumn(
    "age_group",
    F.when(F.col("age") < 20, "Below 20")
     .when(F.col("age") <= 30, "20-30")
     .when(F.col("age") <= 40, "31-40")
     .when(F.col("age") <= 50, "41-50")
     .when(F.col("age") <= 60, "51-60")
     .otherwise("Above 60")
).groupBy("age_group").count().orderBy("age_group")

display(age_analysis)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 85, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6d513791-5a8f-4bd4-b05a-781a516d06d7)

In [84]:
active_analysis = (
    customers_df
    .groupBy("is_active")
    .agg(
        F.count("customer_id").alias("total_customers")
    )
)

display(active_analysis)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 86, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e8dbb1df-85c8-4b29-9e3f-4767e1e41fee)

In [85]:
total_customers = customers_df.count()

active_percentage = customers_df.groupBy("is_active").agg(
    F.count("customer_id").alias("total_customers")
).withColumn(
    "percentage",
    F.round(
        F.col("total_customers") / total_customers * 100, 2
    )
)

display(active_percentage)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 87, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f97bcc45-203f-4f97-b56f-0a4263267b2b)

In [86]:
loyalty_status = (
    customers_df
    .groupBy("loyalty_tier", "is_active")
    .agg(
        F.count("customer_id").alias("total_customers")
    )
    .orderBy("loyalty_tier")
)

display(loyalty_status)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 88, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ea4ba4b4-90d3-4ad6-a62c-76aa13df3609)

In [87]:
signup_analysis = customers_df.withColumn(
    "signup_year",
    F.year("signup_date")
).withColumn(
    "signup_month",
    F.month("signup_date")
)

display(signup_analysis.select(
    "customer_id",
    "signup_date",
    "signup_year",
    "signup_month"
))

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 89, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, efe588f6-4a76-49bc-bd83-0a564c68c792)

In [88]:
yearly_signups = (
    signup_analysis
    .groupBy("signup_year")
    .agg(
        F.countDistinct("customer_id").alias("total_signups")
    )
    .orderBy("signup_year")
)

display(yearly_signups)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 91, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, adc76e49-7445-496d-b05d-daf29b38ce49)

In [89]:
monthly_signups = (
    signup_analysis
    .groupBy("signup_year", "signup_month")
    .agg(
        F.countDistinct("customer_id").alias("total_signups")
    )
    .orderBy("signup_year", "signup_month")
)

display(monthly_signups)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 92, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f0c82fda-2b0b-4db6-9872-7a7e354069e9)

In [90]:
monthly_trends = (
    signup_analysis
    .groupBy("signup_month")
    .agg(
        F.countDistinct("customer_id").alias("total_signups")
    )
    .orderBy(F.desc("total_signups"))
)

display(monthly_trends)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 99, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 709981fe-909b-417a-991c-cdb0f8b647c5)

In [91]:
future_signups = customers_df.filter(
    F.col("signup_date") > F.current_date()
)

print("Future Signup Records:", future_signups.count())

display(future_signups)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 101, Finished, Available, Finished, False)

Future Signup Records: 0


SynapseWidget(Synapse.DataFrame, 520b2284-4765-4452-a034-d60ea13fcf4b)

In [92]:
invalid_birth_dates = customers_df.filter(
    (F.col("birth_date") > F.current_date()) |
    (F.col("birth_date").isNull())
)

print("Invalid Birth Date Records:", invalid_birth_dates.count())

display(invalid_birth_dates)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 102, Finished, Available, Finished, False)

Invalid Birth Date Records: 0


SynapseWidget(Synapse.DataFrame, e53dcab3-ace8-4109-aac9-ec2a139b1bf5)

In [93]:
valid_tiers = ["Bronze", "Silver", "Gold", "Platinum"]

invalid_loyalty = customers_df.filter(
    ~F.col("loyalty_tier").isin(valid_tiers) |
    F.col("loyalty_tier").isNull()
)

print("Invalid Loyalty Records:", invalid_loyalty.count())

display(invalid_loyalty)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 103, Finished, Available, Finished, False)

Invalid Loyalty Records: 0


SynapseWidget(Synapse.DataFrame, fb6677fc-4681-4f91-b63d-1062770a8f86)

In [94]:
# Create cleaned customer dataframe

silver_customers_df = (
    customers_df
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("first_name", F.initcap(F.trim(F.col("first_name"))))
    .withColumn("last_name", F.initcap(F.trim(F.col("last_name"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .dropDuplicates(["customer_id"])
)

display(silver_customers_df)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 105, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 92e4f2a8-8e48-454e-a3dd-661c94cad3fa)

In [95]:
# Final validation of Silver customer data
print("Total Silver Records:", silver_customers_df.count())

print("Distinct Customer IDs:",
      silver_customers_df.select("customer_id").distinct().count())

print("Duplicate Customer IDs:",
      silver_customers_df.count() -
      silver_customers_df.select("customer_id").distinct().count())

print("Total Columns:", len(silver_customers_df.columns))

silver_customers_df.printSchema()

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 108, Finished, Available, Finished, False)

Total Silver Records: 5000
Distinct Customer IDs: 5000
Duplicate Customer IDs: 0
Total Columns: 12
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- age: long (nullable = true)



In [96]:
# Save cleaned customer data into Silver layer

silver_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_customers")

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 109, Finished, Available, Finished, False)

In [97]:
# Verify Silver Delta table

silver_check_df = spark.read.table("silver_customers")

print("Silver Table Records:", silver_check_df.count())

display(silver_check_df.limit(10))

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 110, Finished, Available, Finished, False)

Silver Table Records: 5000


SynapseWidget(Synapse.DataFrame, 76d46613-0e84-43e6-ab7a-d9e2b5782280)

In [99]:
# Find product master file

def find_product_files(path):
    for item in notebookutils.fs.ls(path):
        if item.isDir:
            find_product_files(item.path)
        elif "products.csv" in item.path.lower():
            print(item.path)

find_product_files("Files/")

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 117, Finished, Available, Finished, False)

abfss://a3dec072-153b-40cc-ae81-06c66b7d6cfc@onelake.dfs.fabric.microsoft.com/01c82466-9006-4134-b795-5fce4a6f12ea/Files/project1_novamart-20261107T100452Z-1-001/project1_novamart/master/products.csv


In [103]:
# Load product master data

product_path = "Files/project1_novamart-20261107T100452Z-1-001/project1_novamart/master/products.csv"

products_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(product_path)
)

display(products_df)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 122, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 576c33f8-7c8c-4ed8-849b-5688326efce5)

In [104]:
from pyspark.sql import functions as F

product_null_analysis = products_df.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in products_df.columns
])

display(product_null_analysis)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 124, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 38dd100c-e89e-48a0-a2c8-1af5366381bb)

In [105]:
total_products = products_df.count()

product_null_percentage = products_df.select([
    (
        F.count(
            F.when(F.col(c).isNull(), 1)
        ) / total_products * 100
    ).alias(c)
    for c in products_df.columns
])

display(product_null_percentage)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 125, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e44019f1-2aee-4b2b-a517-be321ba7fa01)

In [106]:
duplicate_products = (
    products_df
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate Product IDs:", duplicate_products.count())

display(duplicate_products)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 126, Finished, Available, Finished, False)

Duplicate Product IDs: 0


SynapseWidget(Synapse.DataFrame, 738b8af2-39ab-427c-b693-e5f051ce09c3)

In [107]:
duplicate_product_names = (
    products_df
    .groupBy("product_name")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

print("Duplicate Product Names:", duplicate_product_names.count())

display(duplicate_product_names)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 128, Finished, Available, Finished, False)

Duplicate Product Names: 32


SynapseWidget(Synapse.DataFrame, 0c442c77-aa23-4384-b17c-06d2201074eb)

In [108]:
invalid_prices = products_df.filter(
    (F.col("mrp") <= 0) |
    (F.col("cost_price") <= 0) |
    (F.col("cost_price") > F.col("mrp"))
)

print("Invalid Price Records:", invalid_prices.count())

display(invalid_prices)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 129, Finished, Available, Finished, False)

Invalid Price Records: 0


SynapseWidget(Synapse.DataFrame, 0e988d5a-805f-4f42-858b-ef203205acf2)

In [109]:
print("Total Categories:", products_df.select("category").distinct().count())

print("Total Subcategories:", products_df.select("sub_category").distinct().count())

print("Total Brands:", products_df.select("brand").distinct().count())

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 130, Finished, Available, Finished, False)

Total Categories: 8
Total Subcategories: 32
Total Brands: 25


In [110]:
category_analysis = (
    products_df
    .groupBy("category")
    .agg(
        F.count("product_id").alias("total_products")
    )
    .orderBy(F.desc("total_products"))
)

display(category_analysis)

StatementMeta(, b4103bab-cb7b-4010-bed7-4fe21a310bdc, 132, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a8dc3062-0798-40c1-99e9-ed252222c2c5)